In [33]:
import sys
import pandas as pd
import numpy as np
import networkx as nx

print("Python:", sys.executable)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("NetworkX:", nx.__version__)

Python: D:\AtmoGraph-Supply-Chain-Ripple-Predictor\.venv\Scripts\python.exe
Pandas: 3.1.1
NumPy: 2.4.6
NetworkX: 3.6.1


In [34]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

from pathlib import Path

# Krish's graph dataset location
DATA_DIR = Path(
    r"D:\AtmoGraph-Supply-Chain-Ripple-Predictor-Krish\graph_database\data"
)

# Project directory
PROJECT_DIR = Path(
    r"D:\AtmoGraph-Supply-Chain-Ripple-Predictor"
)

print("Environment ready!")
print("Dataset directory exists:", DATA_DIR.exists())


Environment ready!
Dataset directory exists: True


In [35]:
# Node datasets
nodes = {
    "suppliers": pd.read_csv(DATA_DIR / "nodes_suppliers.csv"),
    "manufacturers": pd.read_csv(DATA_DIR / "nodes_manufacturers.csv"),
    "ports": pd.read_csv(DATA_DIR / "nodes_ports.csv"),
    "distributors": pd.read_csv(DATA_DIR / "nodes_distributors.csv"),
    "disruption_events": pd.read_csv(DATA_DIR / "nodes_disruption_events.csv")
}

# Relationship datasets
edges = {
    "supplies_to": pd.read_csv(DATA_DIR / "edges_supplies_to.csv"),
    "ships_via": pd.read_csv(DATA_DIR / "edges_ships_via.csv"),
    "route_to": pd.read_csv(DATA_DIR / "edges_route_to.csv"),
    "delivers_to": pd.read_csv(DATA_DIR / "edges_delivers_to.csv"),
    "had_disruption": pd.read_csv(DATA_DIR / "edges_had_disruption.csv")
}

print("All datasets loaded successfully!")
print("Node datasets:", len(nodes))
print("Relationship datasets:", len(edges))

All datasets loaded successfully!
Node datasets: 5
Relationship datasets: 5


In [36]:
print("NODE DATASETS")

for name, df in nodes.items():
    print(f"{name}: {df.shape}")

print("\nRELATIONSHIP DATASETS")

for name, df in edges.items():
    print(f"{name}: {df.shape}")

NODE DATASETS
suppliers: (5, 13)
manufacturers: (5, 13)
ports: (13, 12)
distributors: (10, 8)
disruption_events: (12, 10)

RELATIONSHIP DATASETS
supplies_to: (25, 8)
ships_via: (9, 9)
route_to: (50, 15)
delivers_to: (12, 7)
had_disruption: (12, 3)


In [37]:
for name, df in nodes.items():
    print(f"\n{'='*50}")
    print(f"DATASET: {name.upper()}")
    print("Columns:", df.columns.tolist())
    display(df.head())

for name, df in edges.items():
    print(f"\n{'='*50}")
    print(f"RELATIONSHIP: {name.upper()}")
    print("Columns:", df.columns.tolist())
    display(df.head())


DATASET: SUPPLIERS
Columns: ['id', 'name', 'country', 'country_iso3', 'lat', 'lon', 'primary_city', 'sku_count', 'avg_defect_rate', 'avg_supplier_lead_time_days', 'inspection_fail_share', 'geo_source', 'source']


,id,name,country,country_iso3,lat,lon,primary_city,sku_count,avg_defect_rate,avg_supplier_lead_time_days,inspection_fail_share,geo_source,source
0,SUP_001,Supplier 1,India,IND,22.5726,88.3639,Kolkata,27,1.804,14.78,0.222,derived_modal_city,derived
1,SUP_002,Supplier 2,India,IND,28.6139,77.2090,Delhi,22,2.363,18.55,0.364,derived_modal_city,derived
2,SUP_003,Supplier 3,India,IND,13.0827,80.2707,Chennai,15,2.466,20.13,0.200,derived_modal_city,derived
3,SUP_004,Supplier 4,India,IND,22.5726,88.3639,Kolkata,18,2.337,15.22,0.667,derived_modal_city,derived
4,SUP_005,Supplier 5,India,IND,13.0827,80.2707,Chennai,18,2.665,18.06,0.389,derived_modal_city,derived



DATASET: MANUFACTURERS
Columns: ['id', 'name', 'country', 'country_iso3', 'lat', 'lon', 'city', 'sku_count', 'production_volume', 'avg_manufacturing_lead_time_days', 'avg_manufacturing_cost', 'avg_defect_rate', 'source']


,id,name,country,country_iso3,lat,lon,city,sku_count,production_volume,avg_manufacturing_lead_time_days,avg_manufacturing_cost,avg_defect_rate,source
0,MFG_MUM,Mumbai Manufacturing Hub,India,IND,19.0760,72.8777,Mumbai,22,13160,19.73,36.73,2.122,derived
1,MFG_DEL,Delhi Manufacturing Hub,India,IND,28.6139,77.2090,Delhi,15,8362,13.53,48.88,2.229,derived
2,MFG_BLR,Bangalore Manufacturing Hub,India,IND,12.9716,77.5946,Bangalore,18,7827,11.78,61.99,2.094,derived
3,MFG_MAA,Chennai Manufacturing Hub,India,IND,13.0827,80.2707,Chennai,20,11984,12.65,51.15,2.638,derived
4,MFG_CCU,Kolkata Manufacturing Hub,India,IND,22.5726,88.3639,Kolkata,25,15451,15.00,41.86,2.286,derived



DATASET: PORTS
Columns: ['id', 'name', 'unlocode', 'country', 'country_iso3', 'lat', 'lon', 'hist_disruption_events', 'hist_avg_affected_days', 'hist_max_severity', 'geo_source', 'source']


,id,name,unlocode,country,country_iso3,lat,lon,hist_disruption_events,hist_avg_affected_days,hist_max_severity,geo_source,source
0,PORT_INNSA,Nhava Sheva (JNPT),INNSA,India,IND,18.95,72.95,0,NaN,NaN,approximate_manual,reference
1,PORT_INMUN,Mundra,INMUN,India,IND,22.74,69.70,0,NaN,NaN,approximate_manual,reference
2,PORT_INMAA,Chennai,INMAA,India,IND,13.09,80.29,0,NaN,NaN,approximate_manual,reference
3,PORT_INCCU,Kolkata,INCCU,India,IND,22.54,88.33,0,NaN,NaN,approximate_manual,reference
4,PORT_CNSHA,Shanghai,CNSHA,China,CHN,31.23,121.47,1,5.0,2.0,approximate_manual,derived


... (10 outputs left out)


RELATIONSHIP: DELIVERS_TO
Columns: ['start_id', 'end_id', 'lead_time_days', 'volume', 'transport_mode', 'dependency_weight', 'source']


,start_id,end_id,lead_time_days,volume,transport_mode,dependency_weight,source
0,PORT_CNSHA,DIST_CHN_01,3,127692.82,Road,1.0,simulated
1,PORT_USNYC,DIST_USA_01,3,75325.25,Road,1.0,simulated
2,PORT_JPTYO,DIST_JPN_01,3,138733.93,Road,1.0,simulated
3,PORT_DEHAM,DIST_DEU_01,3,61050.18,Road,1.0,simulated
4,PORT_GBFXT,DIST_GBR_01,3,73024.08,Road,1.0,simulated



RELATIONSHIP: HAD_DISRUPTION
Columns: ['start_id', 'end_id', 'source']


,start_id,end_id,source
0,PORT_CNSHA,EVT_001,real (Port Disruption DB)
1,PORT_USNYC,EVT_002,real (Port Disruption DB)
2,PORT_USNYC,EVT_003,real (Port Disruption DB)
3,PORT_USNYC,EVT_004,real (Port Disruption DB)
4,PORT_USNYC,EVT_005,real (Port Disruption DB)


In [38]:
# Summary of all node and relationship datasets

summary = []

for name, df in nodes.items():
    summary.append({
        "Dataset": name,
        "Type": "Node",
        "Rows": len(df),
        "Columns": len(df.columns)
    })

for name, df in edges.items():
    summary.append({
        "Dataset": name,
        "Type": "Relationship",
        "Rows": len(df),
        "Columns": len(df.columns)
    })

summary_df = pd.DataFrame(summary)

display(summary_df)

,Dataset,Type,Rows,Columns
0,suppliers,Node,5,13
1,manufacturers,Node,5,13
2,ports,Node,13,12
3,distributors,Node,10,8
4,disruption_events,Node,12,10
5,supplies_to,Relationship,25,8
6,ships_via,Relationship,9,9
7,route_to,Relationship,50,15
8,delivers_to,Relationship,12,7
9,had_disruption,Relationship,12,3


In [39]:
for name, df in nodes.items():
    print(f"\n{name.upper()}")
    missing = df.isnull().sum()
    print(missing[missing > 0] if missing.any() else "No missing values")

for name, df in edges.items():
    print(f"\n{name.upper()}")
    missing = df.isnull().sum()
    print(missing[missing > 0] if missing.any() else "No missing values")


SUPPLIERS
No missing values

MANUFACTURERS
No missing values

PORTS
hist_avg_affected_days    9
hist_max_severity         9
dtype: int64

DISTRIBUTORS
No missing values

DISRUPTION_EVENTS
No missing values

SUPPLIES_TO
No missing values

SHIPS_VIA
No missing values

ROUTE_TO
No missing values

DELIVERS_TO
No missing values

HAD_DISRUPTION
No missing values


In [40]:
for name, df in nodes.items():
    print(f"\n{name.upper()}")
    print("Duplicate IDs:", df["id"].duplicated().sum())
    print("Unique IDs:", df["id"].nunique())


SUPPLIERS
Duplicate IDs: 0
Unique IDs: 5

MANUFACTURERS
Duplicate IDs: 0
Unique IDs: 5

PORTS
Duplicate IDs: 0
Unique IDs: 13

DISTRIBUTORS
Duplicate IDs: 0
Unique IDs: 10

DISRUPTION_EVENTS
Duplicate IDs: 0
Unique IDs: 12


In [41]:
# Combine all node IDs
all_node_ids = set()

for df in nodes.values():
    all_node_ids.update(df["id"].astype(str))

# Check relationship endpoints
for name, df in edges.items():

    missing_start = ~df["start_id"].astype(str).isin(all_node_ids)
    missing_end = ~df["end_id"].astype(str).isin(all_node_ids)

    print(f"\n{name.upper()}")
    print("Total relationships:", len(df))
    print("Invalid start IDs:", missing_start.sum())
    print("Invalid end IDs:", missing_end.sum())


SUPPLIES_TO
Total relationships: 25
Invalid start IDs: 0
Invalid end IDs: 0

SHIPS_VIA
Total relationships: 9
Invalid start IDs: 0
Invalid end IDs: 0

ROUTE_TO
Total relationships: 50
Invalid start IDs: 0
Invalid end IDs: 0

DELIVERS_TO
Total relationships: 12
Invalid start IDs: 0
Invalid end IDs: 0

HAD_DISRUPTION
Total relationships: 12
Invalid start IDs: 0
Invalid end IDs: 0


In [42]:
# Display relationship columns and sample records

for name, df in edges.items():
    print(f"\n{'='*60}")
    print(f"RELATIONSHIP: {name.upper()}")
    print("="*60)

    print("Columns:", df.columns.tolist())
    display(df.head(3))


RELATIONSHIP: SUPPLIES_TO
Columns: ['start_id', 'end_id', 'lead_time_days', 'volume', 'transport_mode', 'dependency_weight', 'n_skus', 'source']


,start_id,end_id,lead_time_days,volume,transport_mode,dependency_weight,n_skus,source
0,SUP_001,MFG_BLR,16.00,253,Road,0.3290,5,derived
1,SUP_001,MFG_MAA,16.25,230,Air,0.2074,4,derived
2,SUP_001,MFG_DEL,6.50,270,Air,0.3683,4,derived



RELATIONSHIP: SHIPS_VIA
Columns: ['start_id', 'end_id', 'lead_time_days', 'volume', 'transport_mode', 'csv_modal_mode', 'dependency_weight', 'distance_km', 'source']


,start_id,end_id,lead_time_days,volume,transport_mode,csv_modal_mode,dependency_weight,distance_km,source
0,MFG_MUM,PORT_INNSA,5.55,758,Road,Air,0.7,15.9,simulated
1,MFG_MUM,PORT_INMUN,5.55,325,Road,Air,0.3,524.3,simulated
2,MFG_DEL,PORT_INMUN,5.93,440,Road,Rail,0.6,995.9,simulated



RELATIONSHIP: ROUTE_TO
Columns: ['start_id', 'end_id', 'route_id', 'lead_time_days', 'volume', 'transport_mode', 'distance_km', 'cargo_type', 'avg_delay_days', 'avg_freight_cost_usd', 'disruption_rate', 'delayed_rate', 'avg_port_congestion', 'source', 'dependency_weight']


,start_id,end_id,route_id,lead_time_days,volume,transport_mode,distance_km,cargo_type,avg_delay_days,avg_freight_cost_usd,disruption_rate,delayed_rate,avg_port_congestion,source,dependency_weight
0,PORT_INNSA,PORT_USNYC,R00001,19,8850.984,Rail,8746,Consumer Goods,6.444,4635.075,0.029,0.666,49.708,derived (country route mapped to gateway ports...,0.1855
1,PORT_JPTYO,PORT_CNSHA,R00002,29,22811.299,Air,5420,Agriculture,6.338,3277.016,0.032,0.642,49.222,derived (country route mapped to gateway ports...,0.1775
2,PORT_AUBNE,PORT_CNSHA,R00003,16,6475.579,Rail,12247,Consumer Goods,6.523,6063.867,0.029,0.657,48.437,derived (country route mapped to gateway ports...,0.0658



RELATIONSHIP: DELIVERS_TO
Columns: ['start_id', 'end_id', 'lead_time_days', 'volume', 'transport_mode', 'dependency_weight', 'source']


,start_id,end_id,lead_time_days,volume,transport_mode,dependency_weight,source
0,PORT_CNSHA,DIST_CHN_01,3,127692.82,Road,1.0,simulated
1,PORT_USNYC,DIST_USA_01,3,75325.25,Road,1.0,simulated
2,PORT_JPTYO,DIST_JPN_01,3,138733.93,Road,1.0,simulated



RELATIONSHIP: HAD_DISRUPTION
Columns: ['start_id', 'end_id', 'source']


,start_id,end_id,source
0,PORT_CNSHA,EVT_001,real (Port Disruption DB)
1,PORT_USNYC,EVT_002,real (Port Disruption DB)
2,PORT_USNYC,EVT_003,real (Port Disruption DB)


In [43]:
# Inspect ports with missing historical information

port_missing = nodes["ports"][
    nodes["ports"]["hist_avg_affected_days"].isnull() |
    nodes["ports"]["hist_max_severity"].isnull()
]

display(port_missing)

,id,name,unlocode,country,country_iso3,lat,lon,hist_disruption_events,hist_avg_affected_days,hist_max_severity,geo_source,source
0,PORT_INNSA,Nhava Sheva (JNPT),INNSA,India,IND,18.95,72.95,0,NaN,NaN,approximate_manual,reference
1,PORT_INMUN,Mundra,INMUN,India,IND,22.74,69.70,0,NaN,NaN,approximate_manual,reference
2,PORT_INMAA,Chennai,INMAA,India,IND,13.09,80.29,0,NaN,NaN,approximate_manual,reference
3,PORT_INCCU,Kolkata,INCCU,India,IND,22.54,88.33,0,NaN,NaN,approximate_manual,reference
7,PORT_DEHAM,Hamburg,DEHAM,Germany,DEU,53.54,9.97,0,NaN,NaN,approximate_manual,reference
8,PORT_GBFXT,Felixstowe,GBFXT,United Kingdom,GBR,51.96,1.35,0,NaN,NaN,approximate_manual,reference
9,PORT_FRLEH,Le Havre,FRLEH,France,FRA,49.48,0.11,0,NaN,NaN,approximate_manual,reference
10,PORT_BRSSZ,Santos,BRSSZ,Brazil,BRA,-23.96,-46.30,0,NaN,NaN,approximate_manual,reference
11,PORT_CAVAN,Vancouver,CAVAN,Canada,CAN,49.29,-123.11,0,NaN,NaN,approximate_manual,reference


In [44]:
# Expected node types for each relationship

expected_types = {
    "supplies_to": ("suppliers", "manufacturers"),
    "ships_via": ("manufacturers", "ports"),
    "route_to": ("ports", "ports"),
    "delivers_to": ("ports", "distributors"),
    "had_disruption": ("ports", "disruption_events")
}

# Create ID-to-node-type mapping
id_to_type = {}

for node_type, df in nodes.items():
    for node_id in df["id"].astype(str):
        id_to_type[node_id] = node_type

# Validate relationship types
for relation, (expected_start, expected_end) in expected_types.items():

    df = edges[relation]

    invalid_start = df["start_id"].astype(str).map(
        id_to_type
    ) != expected_start

    invalid_end = df["end_id"].astype(str).map(
        id_to_type
    ) != expected_end

    print(f"\n{relation.upper()}")
    print("Incorrect source types:", invalid_start.sum())
    print("Incorrect destination types:", invalid_end.sum())


SUPPLIES_TO
Incorrect source types: 0
Incorrect destination types: 0

SHIPS_VIA
Incorrect source types: 0
Incorrect destination types: 0

ROUTE_TO
Incorrect source types: 0
Incorrect destination types: 0

DELIVERS_TO
Incorrect source types: 0
Incorrect destination types: 0

HAD_DISRUPTION
Incorrect source types: 0
Incorrect destination types: 0


In [45]:
ships = edges["ships_via"]

mode_mismatch = ships[
    ships["transport_mode"].astype(str).str.strip().str.lower()
    != ships["csv_modal_mode"].astype(str).str.strip().str.lower()
]

print("Total SHIPS_VIA relationships:", len(ships))
print("Mode mismatches:", len(mode_mismatch))

display(mode_mismatch)

Total SHIPS_VIA relationships: 9
Mode mismatches: 5


,start_id,end_id,lead_time_days,volume,transport_mode,csv_modal_mode,dependency_weight,distance_km,source
0,MFG_MUM,PORT_INNSA,5.55,758,Road,Air,0.7,15.9,simulated
1,MFG_MUM,PORT_INMUN,5.55,325,Road,Air,0.3,524.3,simulated
2,MFG_DEL,PORT_INMUN,5.93,440,Road,Rail,0.6,995.9,simulated
3,MFG_DEL,PORT_INNSA,5.93,293,Road,Rail,0.4,1158.4,simulated
6,MFG_MAA,PORT_INMAA,6.00,1109,Road,Air,1.0,2.2,simulated


In [46]:
for name, df in edges.items():

    print(f"\n{name.upper()}")

    if "source" in df.columns:
        print(df["source"].value_counts(dropna=False))
    else:
        print("No source column available")


SUPPLIES_TO
source
derived    25
Name: count, dtype: int64

SHIPS_VIA
source
simulated    9
Name: count, dtype: int64

ROUTE_TO
source
derived (country route mapped to gateway ports = simulated)    50
Name: count, dtype: int64

DELIVERS_TO
source
simulated    12
Name: count, dtype: int64

HAD_DISRUPTION
source
real (Port Disruption DB)    12
Name: count, dtype: int64


In [47]:
# Compare transportation-mode values

ships = edges["ships_via"]

print("Transport mode distribution:")
display(ships["transport_mode"].value_counts())

print("\nOriginal CSV mode distribution:")
display(ships["csv_modal_mode"].value_counts())

print("\nCross-tabulation:")
display(pd.crosstab(
    ships["transport_mode"],
    ships["csv_modal_mode"],
    margins=True
))

print("\nComplete SHIPS_VIA records:")
display(ships)

Transport mode distribution:


transport_mode
Road    9
Name: count, dtype: int64


Original CSV mode distribution:


csv_modal_mode
Road    4
Air     3
Rail    2
Name: count, dtype: int64


Cross-tabulation:


csv_modal_mode,Air,Rail,Road,All
transport_mode,,,,
Road,3,2,4,9
All,3,2,4,9



Complete SHIPS_VIA records:


,start_id,end_id,lead_time_days,volume,transport_mode,csv_modal_mode,dependency_weight,distance_km,source
0,MFG_MUM,PORT_INNSA,5.55,758,Road,Air,0.7,15.9,simulated
1,MFG_MUM,PORT_INMUN,5.55,325,Road,Air,0.3,524.3,simulated
2,MFG_DEL,PORT_INMUN,5.93,440,Road,Rail,0.6,995.9,simulated
3,MFG_DEL,PORT_INNSA,5.93,293,Road,Rail,0.4,1158.4,simulated
4,MFG_BLR,PORT_INMAA,5.28,461,Road,Road,0.6,292.3,simulated
5,MFG_BLR,PORT_INNSA,5.28,308,Road,Road,0.4,829.6,simulated
6,MFG_MAA,PORT_INMAA,6.00,1109,Road,Air,1.0,2.2,simulated
7,MFG_CCU,PORT_INCCU,5.96,982,Road,Road,0.8,5.0,simulated
8,MFG_CCU,PORT_INMAA,5.96,246,Road,Road,0.2,1356.4,simulated


In [48]:
# Inspect disruption event information

events = nodes["disruption_events"]

print("Disruption event columns:")
print(events.columns.tolist())

print("\nDisruption event records:")
display(events)

print("\nEvent source distribution:")

if "source" in events.columns:
    display(events["source"].value_counts(dropna=False))

Disruption event columns:
['id', 'name', 'year', 'reduction_days', 'shutdown_days', 'recovery_days', 'total_affected_days', 'severity', 'source', 'status']

Disruption event records:


,id,name,year,reduction_days,shutdown_days,recovery_days,total_affected_days,severity,source,status
0,EVT_001,Typhoon Lekima,2019,3,0,2,5,2,real (Port Disruption DB),historical
1,EVT_002,Hurricane Irene,2011,2,1,3,6,3,real (Port Disruption DB),historical
2,EVT_003,Hurricane Sandy,2012,1,5,3,9,3,real (Port Disruption DB),historical
3,EVT_004,Hurricane Matthew,2016,2,0,2,4,1,real (Port Disruption DB),historical
4,EVT_005,Hurricane Maria,2017,0,0,0,0,0,real (Port Disruption DB),historical
5,EVT_006,Typhoon Hagibis,2019,3,1,3,7,3,real (Port Disruption DB),historical
6,EVT_007,Typhoon Lingling,2019,0,0,0,0,0,real (Port Disruption DB),historical
7,EVT_008,Tropical Cyclone Oswald,2013,0,0,0,0,0,real (Port Disruption DB),historical
8,EVT_009,Tropical Cyclone Ita,2014,0,0,0,0,0,real (Port Disruption DB),historical
9,EVT_010,Tropical Cyclone Marcia,2015,0,0,0,0,0,real (Port Disruption DB),historical



Event source distribution:


source
real (Port Disruption DB)    12
Name: count, dtype: int64

In [49]:
# Analyze disruption severity and impact

events = nodes["disruption_events"]

print("Severity distribution:")
display(events["severity"].value_counts().sort_index())

print("\nTotal affected days distribution:")
display(events["total_affected_days"].describe())

print("\nEvents with zero impact:")
zero_impact = events[
    (events["total_affected_days"] == 0) &
    (events["severity"] == 0)
]

display(zero_impact)

print("\nEvents with recorded impact:")
recorded_impact = events[
    (events["total_affected_days"] > 0) |
    (events["severity"] > 0)
]

display(recorded_impact)

Severity distribution:


severity
0    6
1    1
2    1
3    4
Name: count, dtype: int64


Total affected days distribution:


count    12.000000
mean      3.000000
std       3.357488
min       0.000000
25%       0.000000
50%       2.000000
75%       5.250000
max       9.000000
Name: total_affected_days, dtype: float64


Events with zero impact:


,id,name,year,reduction_days,shutdown_days,recovery_days,total_affected_days,severity,source,status
4,EVT_005,Hurricane Maria,2017,0,0,0,0,0,real (Port Disruption DB),historical
6,EVT_007,Typhoon Lingling,2019,0,0,0,0,0,real (Port Disruption DB),historical
7,EVT_008,Tropical Cyclone Oswald,2013,0,0,0,0,0,real (Port Disruption DB),historical
8,EVT_009,Tropical Cyclone Ita,2014,0,0,0,0,0,real (Port Disruption DB),historical
9,EVT_010,Tropical Cyclone Marcia,2015,0,0,0,0,0,real (Port Disruption DB),historical
11,EVT_012,Tropical Cyclone Debbie,2017,0,0,0,0,0,real (Port Disruption DB),historical



Events with recorded impact:


,id,name,year,reduction_days,shutdown_days,recovery_days,total_affected_days,severity,source,status
0,EVT_001,Typhoon Lekima,2019,3,0,2,5,2,real (Port Disruption DB),historical
1,EVT_002,Hurricane Irene,2011,2,1,3,6,3,real (Port Disruption DB),historical
2,EVT_003,Hurricane Sandy,2012,1,5,3,9,3,real (Port Disruption DB),historical
3,EVT_004,Hurricane Matthew,2016,2,0,2,4,1,real (Port Disruption DB),historical
5,EVT_006,Typhoon Hagibis,2019,3,1,3,7,3,real (Port Disruption DB),historical
10,EVT_011,Tropical Cyclone Nathan,2015,2,1,2,5,3,real (Port Disruption DB),historical


In [50]:
# Connect disruption events to ports

disruption_links = edges["had_disruption"]

event_details = events[
    ["id", "name", "year", "total_affected_days", "severity"]
]

port_details = nodes["ports"][
    ["id", "name", "country"]
]

event_mapping = (
    disruption_links
    .merge(
        port_details,
        left_on="start_id",
        right_on="id",
        how="left"
    )
    .drop(columns=["id"])
    .rename(columns={"name": "port_name"})
    .merge(
        event_details,
        left_on="end_id",
        right_on="id",
        how="left"
    )
    .drop(columns=["id"])
    .rename(columns={"name": "event_name"})
)

display(event_mapping)

,start_id,end_id,source,port_name,country,event_name,year,total_affected_days,severity
0,PORT_CNSHA,EVT_001,real (Port Disruption DB),Shanghai,China,Typhoon Lekima,2019,5,2
1,PORT_USNYC,EVT_002,real (Port Disruption DB),Port of New York and New Jersey,United States,Hurricane Irene,2011,6,3
2,PORT_USNYC,EVT_003,real (Port Disruption DB),Port of New York and New Jersey,United States,Hurricane Sandy,2012,9,3
3,PORT_USNYC,EVT_004,real (Port Disruption DB),Port of New York and New Jersey,United States,Hurricane Matthew,2016,4,1
4,PORT_USNYC,EVT_005,real (Port Disruption DB),Port of New York and New Jersey,United States,Hurricane Maria,2017,0,0
5,PORT_JPTYO,EVT_006,real (Port Disruption DB),Tokyo,Japan,Typhoon Hagibis,2019,7,3
6,PORT_JPTYO,EVT_007,real (Port Disruption DB),Tokyo,Japan,Typhoon Lingling,2019,0,0
7,PORT_AUBNE,EVT_008,real (Port Disruption DB),Brisbane,Australia,Tropical Cyclone Oswald,2013,0,0
8,PORT_AUBNE,EVT_009,real (Port Disruption DB),Brisbane,Australia,Tropical Cyclone Ita,2014,0,0
9,PORT_AUBNE,EVT_010,real (Port Disruption DB),Brisbane,Australia,Tropical Cyclone Marcia,2015,0,0


In [51]:
# Validate disruption duration calculations

events = nodes["disruption_events"].copy()

events["calculated_affected_days"] = (
    events["reduction_days"]
    + events["shutdown_days"]
    + events["recovery_days"]
)

events["duration_matches"] = (
    events["calculated_affected_days"]
    == events["total_affected_days"]
)

print("Duration mismatches:",
      (~events["duration_matches"]).sum())

display(events[
    [
        "id",
        "name",
        "reduction_days",
        "shutdown_days",
        "recovery_days",
        "total_affected_days",
        "calculated_affected_days",
        "duration_matches"
    ]
])

Duration mismatches: 0


,id,name,reduction_days,shutdown_days,recovery_days,total_affected_days,calculated_affected_days,duration_matches
0,EVT_001,Typhoon Lekima,3,0,2,5,5,True
1,EVT_002,Hurricane Irene,2,1,3,6,6,True
2,EVT_003,Hurricane Sandy,1,5,3,9,9,True
3,EVT_004,Hurricane Matthew,2,0,2,4,4,True
4,EVT_005,Hurricane Maria,0,0,0,0,0,True
5,EVT_006,Typhoon Hagibis,3,1,3,7,7,True
6,EVT_007,Typhoon Lingling,0,0,0,0,0,True
7,EVT_008,Tropical Cyclone Oswald,0,0,0,0,0,True
8,EVT_009,Tropical Cyclone Ita,0,0,0,0,0,True
9,EVT_010,Tropical Cyclone Marcia,0,0,0,0,0,True


In [52]:
# Count events connected to each port

port_event_counts = (
    edges["had_disruption"]
    .groupby("start_id")
    .size()
    .reset_index(name="event_count")
)

port_event_counts = port_event_counts.merge(
    nodes["ports"][["id", "name", "country"]],
    left_on="start_id",
    right_on="id",
    how="left"
)

port_event_counts = port_event_counts.drop(columns=["id"])

display(port_event_counts)

,start_id,event_count,name,country
0,PORT_AUBNE,5,Brisbane,Australia
1,PORT_CNSHA,1,Shanghai,China
2,PORT_JPTYO,2,Tokyo,Japan
3,PORT_USNYC,4,Port of New York and New Jersey,United States


In [53]:
# Complete feature inventory

for name, df in nodes.items():

    print(f"\n{'='*65}")
    print(f"NODE DATASET: {name.upper()}")
    print("="*65)

    inventory = pd.DataFrame({
        "Column": df.columns,
        "Data Type": df.dtypes.astype(str).values,
        "Unique Values": [
            df[col].nunique() for col in df.columns
        ],
        "Missing Values": df.isnull().sum().values
    })

    display(inventory)


NODE DATASET: SUPPLIERS


,Column,Data Type,Unique Values,Missing Values
0,id,str,5,0
1,name,str,5,0
2,country,str,1,0
3,country_iso3,str,1,0
4,lat,float64,3,0
5,lon,float64,3,0
6,primary_city,str,3,0
7,sku_count,int64,4,0
8,avg_defect_rate,float64,5,0
9,avg_supplier_lead_time_days,float64,5,0



NODE DATASET: MANUFACTURERS


,Column,Data Type,Unique Values,Missing Values
0,id,str,5,0
1,name,str,5,0
2,country,str,1,0
3,country_iso3,str,1,0
4,lat,float64,5,0
5,lon,float64,5,0
6,city,str,5,0
7,sku_count,int64,5,0
8,production_volume,int64,5,0
9,avg_manufacturing_lead_time_days,float64,5,0



NODE DATASET: PORTS


,Column,Data Type,Unique Values,Missing Values
0,id,str,13,0
1,name,str,13,0
2,unlocode,str,13,0
3,country,str,10,0
4,country_iso3,str,10,0
5,lat,float64,13,0
6,lon,float64,13,0
7,hist_disruption_events,int64,5,0
8,hist_avg_affected_days,float64,4,9
9,hist_max_severity,float64,2,9



NODE DATASET: DISTRIBUTORS


,Column,Data Type,Unique Values,Missing Values
0,id,str,10,0
1,name,str,10,0
2,country,str,10,0
3,country_iso3,str,10,0
4,lat,float64,10,0
5,lon,float64,10,0
6,geo_source,str,1,0
7,source,str,1,0



NODE DATASET: DISRUPTION_EVENTS


,Column,Data Type,Unique Values,Missing Values
0,id,str,12,0
1,name,str,12,0
2,year,int64,8,0
3,reduction_days,int64,4,0
4,shutdown_days,int64,3,0
5,recovery_days,int64,3,0
6,total_affected_days,int64,6,0
7,severity,int64,4,0
8,source,str,1,0
9,status,str,1,0


In [54]:
# Relationship feature inventory

for name, df in edges.items():

    print(f"\n{'='*65}")
    print(f"RELATIONSHIP DATASET: {name.upper()}")
    print("="*65)

    inventory = pd.DataFrame({
        "Column": df.columns,
        "Data Type": df.dtypes.astype(str).values,
        "Unique Values": [
            df[col].nunique() for col in df.columns
        ],
        "Missing Values": df.isnull().sum().values
    })

    display(inventory)


RELATIONSHIP DATASET: SUPPLIES_TO


,Column,Data Type,Unique Values,Missing Values
0,start_id,str,5,0
1,end_id,str,5,0
2,lead_time_days,float64,24,0
3,volume,int64,25,0
4,transport_mode,str,4,0
5,dependency_weight,float64,25,0
6,n_skus,int64,7,0
7,source,str,1,0



RELATIONSHIP DATASET: SHIPS_VIA


,Column,Data Type,Unique Values,Missing Values
0,start_id,str,5,0
1,end_id,str,4,0
2,lead_time_days,float64,5,0
3,volume,int64,9,0
4,transport_mode,str,1,0
5,csv_modal_mode,str,3,0
6,dependency_weight,float64,7,0
7,distance_km,float64,9,0
8,source,str,1,0



RELATIONSHIP DATASET: ROUTE_TO


,Column,Data Type,Unique Values,Missing Values
0,start_id,str,13,0
1,end_id,str,12,0
2,route_id,str,50,0
3,lead_time_days,int64,31,0
4,volume,float64,50,0
5,transport_mode,str,4,0
6,distance_km,int64,50,0
7,cargo_type,str,5,0
8,avg_delay_days,float64,48,0
9,avg_freight_cost_usd,float64,50,0



RELATIONSHIP DATASET: DELIVERS_TO


,Column,Data Type,Unique Values,Missing Values
0,start_id,str,12,0
1,end_id,str,10,0
2,lead_time_days,int64,2,0
3,volume,float64,12,0
4,transport_mode,str,1,0
5,dependency_weight,float64,4,0
6,source,str,1,0



RELATIONSHIP DATASET: HAD_DISRUPTION


,Column,Data Type,Unique Values,Missing Values
0,start_id,str,4,0
1,end_id,str,12,0
2,source,str,1,0


In [55]:
# Identify constant and variable columns

for name, df in nodes.items():

    print(f"\n{'='*55}")
    print(f"NODE: {name.upper()}")
    print("="*55)

    for col in df.columns:
        unique_count = df[col].nunique(dropna=False)

        if col != "id":
            status = "CONSTANT" if unique_count <= 1 else "VARIABLE"
            print(f"{col:35} {unique_count:5}  {status}")


NODE: SUPPLIERS
name                                    5  VARIABLE
country                                 1  CONSTANT
country_iso3                            1  CONSTANT
lat                                     3  VARIABLE
lon                                     3  VARIABLE
primary_city                            3  VARIABLE
sku_count                               4  VARIABLE
avg_defect_rate                         5  VARIABLE
avg_supplier_lead_time_days             5  VARIABLE
inspection_fail_share                   5  VARIABLE
geo_source                              1  CONSTANT
source                                  1  CONSTANT

NODE: MANUFACTURERS
name                                    5  VARIABLE
country                                 1  CONSTANT
country_iso3                            1  CONSTANT
lat                                     5  VARIABLE
lon                                     5  VARIABLE
city                                    5  VARIABLE
sku_count                 

In [56]:
# Compare historical port event counts with graph connections

port_event_counts = (
    edges["had_disruption"]
    .groupby("start_id")
    .size()
)

port_check = nodes["ports"][
    ["id", "name", "hist_disruption_events"]
].copy()

port_check["graph_event_count"] = (
    port_check["id"]
    .map(port_event_counts)
    .fillna(0)
    .astype(int)
)

port_check["count_matches"] = (
    port_check["hist_disruption_events"]
    == port_check["graph_event_count"]
)

display(port_check)

print(
    "Count mismatches:",
    (~port_check["count_matches"]).sum()
)

,id,name,hist_disruption_events,graph_event_count,count_matches
0,PORT_INNSA,Nhava Sheva (JNPT),0,0,True
1,PORT_INMUN,Mundra,0,0,True
2,PORT_INMAA,Chennai,0,0,True
3,PORT_INCCU,Kolkata,0,0,True
4,PORT_CNSHA,Shanghai,1,1,True
5,PORT_USNYC,Port of New York and New Jersey,4,4,True
6,PORT_JPTYO,Tokyo,2,2,True
7,PORT_DEHAM,Hamburg,0,0,True
8,PORT_GBFXT,Felixstowe,0,0,True
9,PORT_FRLEH,Le Havre,0,0,True


Count mismatches: 0


In [57]:

# Cell 24: Relationship Feature Distribution

for name, df in edges.items():
    print(f"\n{'=' * 60}")
    print(f"RELATIONSHIP: {name.upper()}")
    print("=" * 60)

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

    if numeric_cols:
        display(df[numeric_cols].describe().T)
    else:
        print("No numerical features found.")



RELATIONSHIP: SUPPLIES_TO


,count,mean,std,min,25%,50%,75%,max
lead_time_days,25.0,17.632,6.218341,3.0000,15.2500,17.5000,22.330,28.0000
volume,25.0,196.880,94.956709,9.0000,140.0000,194.0000,270.000,406.0000
dependency_weight,25.0,0.200,0.098201,0.0123,0.1173,0.1994,0.248,0.4311
n_skus,25.0,4.000,1.581139,1.0000,3.0000,4.0000,5.000,8.0000



RELATIONSHIP: SHIPS_VIA


,count,mean,std,min,25%,50%,75%,max
lead_time_days,9.0,5.715556,0.301376,5.28,5.55,5.93,5.96,6.0
volume,9.0,546.888889,322.094103,246.00,308.00,440.00,758.00,1109.0
dependency_weight,9.0,0.555556,0.255495,0.20,0.40,0.60,0.70,1.0
distance_km,9.0,575.555556,529.268262,2.20,15.90,524.30,995.90,1356.4



RELATIONSHIP: ROUTE_TO


,count,mean,std,min,25%,50%,75%,max
lead_time_days,50.0,25.120000,12.266829,3.0000,16.000000,26.5000,35.500000,45.000
volume,50.0,18823.910600,8899.590367,3952.8170,12086.247500,16817.3895,23346.465250,36395.442
distance_km,50.0,7333.600000,3865.000795,564.0000,4921.500000,7652.5000,10278.500000,14048.000
avg_delay_days,50.0,6.381020,0.094337,6.1820,6.315750,6.4010,6.443750,6.589
avg_freight_cost_usd,50.0,4058.009080,1575.963443,1299.5060,3076.267500,4188.6690,5259.968250,6795.879
disruption_rate,50.0,0.031960,0.006797,0.0180,0.027000,0.0320,0.036500,0.046
delayed_rate,50.0,0.640140,0.017855,0.6050,0.628000,0.6380,0.650000,0.682
avg_port_congestion,50.0,49.697800,1.127773,46.4370,48.924000,49.7960,50.558000,51.511
dependency_weight,50.0,0.260008,0.203009,0.0471,0.131775,0.1936,0.303675,1.000



RELATIONSHIP: DELIVERS_TO


,count,mean,std,min,25%,50%,75%,max
lead_time_days,12.0,2.750000,0.452267,2.0000,2.750000,3.00,3.000,3.00
volume,12.0,78432.961667,51348.814989,18572.2200,44698.267500,67037.13,115844.635,180751.67
dependency_weight,12.0,0.833333,0.301934,0.2932,0.841825,1.00,1.000,1.00



RELATIONSHIP: HAD_DISRUPTION
No numerical features found.


In [58]:

# Cell 25: Potential Outlier Analysis

for name, df in edges.items():
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

    print(f"\n{'=' * 60}")
    print(f"OUTLIERS: {name.upper()}")
    print("=" * 60)

    for col in numeric_cols:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        count = ((df[col] < lower) | (df[col] > upper)).sum()

        if count > 0:
            print(f"{col}: {count} potential outliers")



OUTLIERS: SUPPLIES_TO
lead_time_days: 1 potential outliers

OUTLIERS: SHIPS_VIA

OUTLIERS: ROUTE_TO
avg_port_congestion: 1 potential outliers
dependency_weight: 3 potential outliers

OUTLIERS: DELIVERS_TO
lead_time_days: 3 potential outliers
dependency_weight: 3 potential outliers

OUTLIERS: HAD_DISRUPTION


In [59]:

# Cell 26: Inspect Potential Outlier Records

outlier_columns = {
    "supplies_to": ["lead_time_days"],
    "route_to": ["avg_port_congestion", "dependency_weight"]
}

for name, columns in outlier_columns.items():
    df = edges[name]

    print(f"\n{'=' * 65}")
    print(f"INSPECTING: {name.upper()}")
    print("=" * 65)

    for col in columns:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        unusual = df[(df[col] < lower) | (df[col] > upper)]

        if not unusual.empty:
            print(f"\nColumn: {col}")
            display(unusual)



INSPECTING: SUPPLIES_TO

Column: lead_time_days


,start_id,end_id,lead_time_days,volume,transport_mode,dependency_weight,n_skus,source
17,SUP_004,MFG_DEL,3.0,86,Rail,0.1173,2,derived



INSPECTING: ROUTE_TO

Column: avg_port_congestion


,start_id,end_id,route_id,lead_time_days,volume,transport_mode,distance_km,cargo_type,avg_delay_days,avg_freight_cost_usd,disruption_rate,delayed_rate,avg_port_congestion,source,dependency_weight
15,PORT_USNYC,PORT_FRLEH,R00016,34,3952.817,Rail,11863,Energy,6.273,5902.338,0.042,0.605,46.437,derived (country route mapped to gateway ports...,0.0606



Column: dependency_weight


,start_id,end_id,route_id,lead_time_days,volume,transport_mode,distance_km,cargo_type,avg_delay_days,avg_freight_cost_usd,disruption_rate,delayed_rate,avg_port_congestion,source,dependency_weight
7,PORT_INMAA,PORT_CAVAN,R00008,31,35574.539,Rail,13710,Technology,6.441,6658.660,0.042,0.628,49.738,derived (country route mapped to gateway ports...,1.0000
17,PORT_INMUN,PORT_CNSHA,R00018,18,26291.520,Sea,647,Technology,6.387,1332.185,0.030,0.650,48.864,derived (country route mapped to gateway ports...,0.7286
23,PORT_INCCU,PORT_FRLEH,R00024,45,15630.214,Rail,13578,Consumer Goods,6.450,6605.051,0.034,0.637,49.442,derived (country route mapped to gateway ports...,1.0000


In [60]:

# Cell 27: Check Zero and Unusually Small Values

print("DELIVERS_TO: Zero-volume records")
display(edges["delivers_to"][edges["delivers_to"]["volume"] == 0])

print("\nSHIPS_VIA: Short-distance records")
display(edges["ships_via"][edges["ships_via"]["distance_km"] < 10])


DELIVERS_TO: Zero-volume records


,start_id,end_id,lead_time_days,volume,transport_mode,dependency_weight,source



SHIPS_VIA: Short-distance records


,start_id,end_id,lead_time_days,volume,transport_mode,csv_modal_mode,dependency_weight,distance_km,source
6,MFG_MAA,PORT_INMAA,6.00,1109,Road,Air,1.0,2.2,simulated
7,MFG_CCU,PORT_INCCU,5.96,982,Road,Road,0.8,5.0,simulated


In [61]:

# Cell 28: Inspect Route R00024

route = edges["route_to"]

display(
    route.loc[
        route["route_id"] == "R00024"
    ].T
)

print("\nMissing values in R00024:")
display(
    route.loc[
        route["route_id"] == "R00024"
    ].isna()
)


,23
start_id,PORT_INCCU
end_id,PORT_FRLEH
route_id,R00024
lead_time_days,45
volume,15630.214
transport_mode,Rail
distance_km,13578
cargo_type,Consumer Goods
avg_delay_days,6.45
avg_freight_cost_usd,6605.051



Missing values in R00024:


,start_id,end_id,route_id,lead_time_days,volume,transport_mode,distance_km,cargo_type,avg_delay_days,avg_freight_cost_usd,disruption_rate,delayed_rate,avg_port_congestion,source,dependency_weight
23,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [62]:

# Cell 29: Recheck Relationship Missing Values

for name, df in edges.items():
    print(f"\n{name.upper()}")
    
    missing = df.isnull().sum()
    missing = missing[missing > 0]

    if missing.empty:
        print("No missing values")
    else:
        display(missing)



SUPPLIES_TO
No missing values

SHIPS_VIA
No missing values

ROUTE_TO
No missing values

DELIVERS_TO
No missing values

HAD_DISRUPTION
No missing values


In [63]:

# Cell 30: Inspect Zero-Volume Delivery

display(
    edges["delivers_to"].loc[
        edges["delivers_to"]["volume"] == 0
    ].T
)


""
start_id
end_id
lead_time_days
volume
transport_mode
dependency_weight
source


In [64]:

# Verify updated graph dataset

for name, df in nodes.items():
    print(f"{name}: {len(df)} nodes")

print("\nRelationship counts:")

total_edges = 0

for name, df in edges.items():
    print(f"{name}: {len(df)} relationships")
    total_edges += len(df)

print("\nTotal nodes:", sum(len(df) for df in nodes.values()))
print("Total relationships:", total_edges)


suppliers: 5 nodes
manufacturers: 5 nodes
ports: 13 nodes
distributors: 10 nodes
disruption_events: 12 nodes

Relationship counts:
supplies_to: 25 relationships
ships_via: 9 relationships
route_to: 50 relationships
delivers_to: 12 relationships
had_disruption: 12 relationships

Total nodes: 45
Total relationships: 108
